In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, lit, array, struct

spark = SparkSession.builder.appName("M5_Analysis_Preparation").getOrCreate()


sales_df = spark.read.csv("sales_train_evaluation.csv", header=True, inferSchema=True)
calendar_df = spark.read.csv("calendar.csv", header=True, inferSchema=True)
prices_df = spark.read.csv("sell_prices.csv", header=True, inferSchema=True)

In [4]:
print(f"tamaño en Calendario: {(calendar_df.count(), len(calendar_df.columns))}")
print(f"tamaño en Precios: {(prices_df.count(), len(prices_df.columns))}")
print(f"tamaño en Ventas: {(sales_df.count(), len(sales_df.columns))}")

tamaño en Calendario: (1969, 14)
tamaño en Precios: (6841121, 4)
tamaño en Ventas: (30490, 1947)


In [5]:
calendar_df.createOrReplaceTempView("calendario")
prices_df.createOrReplaceTempView("precios")
sales_df.createOrReplaceTempView("ventas_crudas")


spark.sql("""
    SELECT state_id, COUNT(DISTINCT store_id) as total_tiendas
    FROM ventas_crudas
    GROUP BY state_id
""").show()

spark.sql("""
    SELECT dept_id, COUNT(DISTINCT item_id) as total_productos
    FROM ventas_crudas
    GROUP BY dept_id
""").show()

spark.sql("""
    SELECT COUNT(DISTINCT item_id) as total_productos_unicos
    FROM ventas_crudas
""").show()

spark.sql("""
    SELECT
        COUNT(*) as total_dias,
        SUM(CASE WHEN event_name_1 IS NULL THEN 1 ELSE 0 END) as dias_sin_evento
    FROM calendario
""").show()

+--------+-------------+
|state_id|total_tiendas|
+--------+-------------+
|      WI|            3|
|      CA|            4|
|      TX|            3|
+--------+-------------+

+-----------+---------------+
|    dept_id|total_productos|
+-----------+---------------+
|    FOODS_3|            823|
|  HOBBIES_1|            416|
|  HOBBIES_2|            149|
|HOUSEHOLD_1|            532|
|    FOODS_1|            216|
|HOUSEHOLD_2|            515|
|    FOODS_2|            398|
+-----------+---------------+

+----------------------+
|total_productos_unicos|
+----------------------+
|                  3049|
+----------------------+

+----------+---------------+
|total_dias|dias_sin_evento|
+----------+---------------+
|      1969|           1807|
+----------+---------------+



In [6]:
columnas_id = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
todas_las_columnas = sales_df.columns
columnas_dias = [col for col in todas_las_columnas if col.startswith('d_')]
ventas_largas_df = sales_df.unpivot(
    ids=columnas_id,
    values=columnas_dias,
    variableColumnName="dia",
    valueColumnName="ventas"
)

In [7]:
print(f"Filas en Ventas: {ventas_largas_df.count()}")

Filas en Ventas: 59181090


In [8]:

calendario_reducido = calendar_df.select("d", "date", "wm_yr_wk", "wday", "event_name_1")

ventas_fechas_df = ventas_largas_df.join(
    calendario_reducido,
    ventas_largas_df.dia == calendario_reducido.d,
    how="left"
).drop("d")


ventas_base_df = ventas_fechas_df.join(
    prices_df,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

ventas_limpias_df = ventas_base_df.where("sell_price IS NOT NULL")

In [9]:
ventas_limpias_df.createOrReplaceTempView("tabla_maestra")

query_clasificacion = """
    SELECT

        date as fecha,
        wm_yr_wk as semana_fiscal,
        CASE
            WHEN wday IN (1, 2) THEN 'Fin de Semana'
            ELSE 'Entre Semana'
        END as tipo_dia,

        event_name_1 as evento_especial,
        item_id as producto,
        dept_id as departamento,
        cat_id as categoria,
        store_id as tienda,
        state_id as estado,
        ventas as unidades_vendidas,
        sell_price as precio_unitario,
        (ventas * sell_price) as ingreso_total,


        CASE
            WHEN sell_price >= 10.0 THEN 'Premium (>=$10)'
            WHEN sell_price >= 5.0 THEN 'Medio ($5-$9.99)'
            ELSE 'Económico (<$5)'
        END as segmento_precio

    FROM tabla_maestra
"""

ventas_clasificadas_df = spark.sql(query_clasificacion)
ventas_clasificadas_df.show(5)

+----------+-------------+-------------+---------------+-----------+------------+---------+------+------+-----------------+---------------+-------------+---------------+
|     fecha|semana_fiscal|     tipo_dia|evento_especial|   producto|departamento|categoria|tienda|estado|unidades_vendidas|precio_unitario|ingreso_total|segmento_precio|
+----------+-------------+-------------+---------------+-----------+------------+---------+------+------+-----------------+---------------+-------------+---------------+
|2011-12-24|        11148|Fin de Semana|           NULL|FOODS_1_001|     FOODS_1|    FOODS|  CA_1|    CA|                1|            2.0|          2.0|Económico (<$5)|
|2011-12-25|        11148|Fin de Semana|      Christmas|FOODS_1_001|     FOODS_1|    FOODS|  CA_1|    CA|                0|            2.0|          0.0|Económico (<$5)|
|2011-12-26|        11148| Entre Semana|           NULL|FOODS_1_001|     FOODS_1|    FOODS|  CA_1|    CA|                1|            2.0|          2

In [11]:
from pyspark.sql import functions as F

auditoria_df = ventas_clasificadas_df.select(
    F.sum(F.when(F.col("unidades_vendidas") < 0, 1).otherwise(0)).alias("total_ventas_negativas"),
    F.sum(F.when(F.col("unidades_vendidas").isNull(), 1).otherwise(0)).alias("ventas_nulas"),
    F.max("unidades_vendidas").alias("venta_diaria_maxima_absoluta")
)

auditoria_df.show()

+----------------------+------------+----------------------------+
|total_ventas_negativas|ventas_nulas|venta_diaria_maxima_absoluta|
+----------------------+------------+----------------------------+
|                     0|           0|                         763|
+----------------------+------------+----------------------------+



In [12]:
ventas_clasificadas_df.createOrReplaceTempView("tabla_clasificada")

query_agregada = """
    SELECT
        fecha,
        semana_fiscal,
        tipo_dia,
        evento_especial,
        departamento,
        categoria,
        tienda,
        estado,
        segmento_precio,
        SUM(unidades_vendidas) as total_unidades,
        SUM(ingreso_total) as total_ingresos
    FROM tabla_clasificada -- ¡Aquí estaba el cambio!
    GROUP BY
        fecha, semana_fiscal, tipo_dia, evento_especial,
        departamento, categoria, tienda, estado, segmento_precio
"""

ventas_resumidas_df = spark.sql(query_agregada)
ventas_resumidas_df.coalesce(1).write.mode("overwrite").csv("dataset_m5_resumido.csv", header=True)